# Sports Futures Simulation Engine

This notebook demonstrates how to use the trained world model to simulate sports seasons
and evaluate futures bets.

## Simulation Workflow

1. Load trained world model (low perplexity = good understanding)
2. Define current season state (wins, losses, remaining schedule)
3. Run N Monte Carlo simulations (e.g., 10,000)
4. Aggregate results to get probability distributions
5. Compare to betting market odds
6. Identify +EV opportunities

## Example: "Will Lakers Make Playoffs?"

- Current record: 35-25 (60 games played)
- Remaining games: 22
- Market odds: +200 (implied 33% probability)
- **Question**: Does our model think they have >33% chance?

In [ ]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from models.world_model import SportsWorldModel
from models.event_tokenizer import SportsEventTokenizer
from simulation.engine import FuturesSimulator, SeasonState, TeamState

# Plotting setup
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Trained Model & Tokenizer

After training in notebook `03_model_training.ipynb`, we should have a checkpoint with low perplexity.

In [ ]:
# Load configuration
with open('../config/model_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Model: {config['model']['d_model']}d, {config['model']['n_layers']} layers")
print(f"  Simulations: {config['simulation']['n_simulations']:,}")

In [ ]:
# Load tokenizer
tokenizer_path = '../data/processed/tokenizer.json'

if Path(tokenizer_path).exists():
    tokenizer = SportsEventTokenizer.load(tokenizer_path)
    print(f"Tokenizer loaded: {tokenizer.vocab_size} tokens")
else:
    print("⚠️ Tokenizer not found. Run notebook 02_event_tokenization.ipynb first.")
    tokenizer = SportsEventTokenizer()  # Use default for demo

In [ ]:
# Load trained model
checkpoint_path = '../models/checkpoints/best_model.pt'

if Path(checkpoint_path).exists():
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Create model
    model = SportsWorldModel(
        vocab_size=tokenizer.vocab_size,
        **config['model']
    )
    
    # Load weights
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Get training stats
    train_perplexity = checkpoint.get('train_perplexity', 'N/A')
    val_perplexity = checkpoint.get('val_perplexity', 'N/A')
    
    print(f"✓ Model loaded from {checkpoint_path}")
    print(f"  Training perplexity: {train_perplexity}")
    print(f"  Validation perplexity: {val_perplexity}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
else:
    print("⚠️ Model checkpoint not found. Run notebook 03_model_training.ipynb first.")
    print("Creating untrained model for demonstration...")
    model = SportsWorldModel(
        vocab_size=tokenizer.vocab_size,
        **config['model']
    )

model.eval()  # Set to evaluation mode

## 2. Define Current Season State

Set up the current standings and remaining schedule.

In [ ]:
# Example: Mid-season state (60 games played, 22 remaining)
# This would normally be loaded from real standings

teams = {
    'LAL': TeamState(
        team_id='LAL',
        wins=35,
        losses=25,
        games_remaining=22,
        strength_rating=0.58,  # Model's learned strength
        injuries=['player_1'],
        recent_form=[1, 1, 0, 1, 0, 1, 1, 1, 0, 1]  # Last 10 games
    ),
    'GSW': TeamState(
        team_id='GSW',
        wins=40,
        losses=20,
        games_remaining=22,
        strength_rating=0.67,
        injuries=[],
        recent_form=[1, 1, 1, 1, 1, 0, 1, 1, 1, 1]
    ),
    # Add more teams...
}

season_state = SeasonState(
    teams=teams,
    games_played=60,
    games_remaining=22,
    standings=pd.DataFrame([
        {'team': 'GSW', 'wins': 40, 'losses': 20, 'win_pct': 0.667},
        {'team': 'LAL', 'wins': 35, 'losses': 25, 'win_pct': 0.583},
        # More teams...
    ])
)

print("Current Standings:")
print(season_state.standings)

## 3. Create Simulator and Run Monte Carlo Simulations

In [ ]:
# Create simulator
simulator = FuturesSimulator(model, tokenizer)

print("Futures Simulator initialized")
print(f"Device: {simulator.device}")

In [ ]:
# Run simulations (this may take several minutes for 10K simulations)
n_sims = config['simulation']['n_simulations']

print(f"Running {n_sims:,} Monte Carlo simulations...")
print("This may take 5-10 minutes depending on your hardware.")

simulation_results = simulator.simulate_season(
    season_state,
    n_simulations=n_sims,
    temperature=config['simulation']['temperature'],
    progress_bar=True
)

print(f"✓ {len(simulation_results):,} simulations complete!")

## 4. Analyze Simulation Results

In [ ]:
# Extract playoff appearance rates
playoff_counts = {}

for sim in simulation_results:
    for team in sim.playoff_teams:
        playoff_counts[team] = playoff_counts.get(team, 0) + 1

# Convert to probabilities
playoff_probs = {team: count / len(simulation_results) for team, count in playoff_counts.items()}

# Sort by probability
playoff_df = pd.DataFrame([
    {'team': team, 'playoff_prob': prob}
    for team, prob in sorted(playoff_probs.items(), key=lambda x: -x[1])
])

print("\nPlayoff Probabilities (Model):")
print(playoff_df.head(10))

In [ ]:
# Visualize playoff probabilities
plt.figure(figsize=(12, 6))
plt.barh(playoff_df['team'][:16], playoff_df['playoff_prob'][:16])
plt.xlabel('Playoff Probability')
plt.title(f'Playoff Probabilities ({n_sims:,} Simulations)')
plt.axvline(0.5, color='red', linestyle='--', label='50% threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Championship probabilities
champion_counts = {}

for sim in simulation_results:
    champion_counts[sim.champion] = champion_counts.get(sim.champion, 0) + 1

champion_probs = {team: count / len(simulation_results) for team, count in champion_counts.items()}

champion_df = pd.DataFrame([
    {'team': team, 'championship_prob': prob}
    for team, prob in sorted(champion_probs.items(), key=lambda x: -x[1])
])

print("\nChampionship Probabilities (Model):")
print(champion_df.head(10))

## 5. Evaluate Futures Bets

Compare model probabilities to betting market odds to find +EV opportunities.

In [ ]:
# Example: Evaluate Lakers playoff bet
# Market odds: +200 (implied 33% probability)
# Model probability: ?

bet_evaluation = simulator.evaluate_futures_bet(
    bet_type="playoff_appearance",
    entity="LAL",
    simulations=simulation_results,
    market_odds=+200  # American odds
)

print("\n" + "="*50)
print("BET EVALUATION: Lakers to Make Playoffs")
print("="*50)
print(f"Market Odds: +200 (implied {bet_evaluation['market_probability']:.1%})")
print(f"Model Probability: {bet_evaluation['model_probability']:.1%}")
print(f"Edge: {bet_evaluation['edge']:.1%}")
print(f"Expected Value: ${bet_evaluation['expected_value']:.2f} (per $100 bet)")
print(f"Kelly Bet Size: {bet_evaluation['kelly_fraction']:.1%} of bankroll")
print(f"Confidence Interval: {bet_evaluation['confidence'][0]:.1%} - {bet_evaluation['confidence'][1]:.1%}")
print(f"\nRecommendation: {bet_evaluation['recommendation']}")
print("="*50)

In [ ]:
# Scan multiple bets to find best opportunities
# (This would use real market odds from The Odds API)

hypothetical_market_odds = {
    ('playoff_appearance', 'LAL'): +200,
    ('playoff_appearance', 'GSW'): -400,
    ('champion', 'LAL'): +800,
    ('champion', 'GSW'): +300,
}

bet_opportunities = []

for (bet_type, entity), odds in hypothetical_market_odds.items():
    try:
        eval_result = simulator.evaluate_futures_bet(
            bet_type=bet_type,
            entity=entity,
            simulations=simulation_results,
            market_odds=odds
        )
        bet_opportunities.append(eval_result)
    except Exception as e:
        print(f"Could not evaluate {bet_type} for {entity}: {e}")

# Sort by edge
bet_opportunities.sort(key=lambda x: x['edge'], reverse=True)

print("\n" + "="*70)
print("TOP BETTING OPPORTUNITIES (Sorted by Edge)")
print("="*70)

for bet in bet_opportunities:
    if bet['recommendation'] == 'BET':
        print(f"\n{bet['bet_type'].upper()}: {bet['entity']}")
        print(f"  Edge: {bet['edge']:+.1%}")
        print(f"  EV: ${bet['expected_value']:+.2f} per $100")
        print(f"  Kelly: {bet['kelly_fraction']:.1%}")

## 6. Win Total Analysis

Analyze the distribution of final win totals for over/under bets.

In [ ]:
# Extract win distributions for a specific team
team = 'LAL'

win_totals = [
    sim.final_standings[sim.final_standings['team'] == team]['wins'].values[0]
    for sim in simulation_results
    if team in sim.final_standings['team'].values
]

# Plot distribution
plt.figure(figsize=(12, 6))
plt.hist(win_totals, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(win_totals), color='red', linestyle='--', label=f'Mean: {np.mean(win_totals):.1f}')
plt.axvline(np.median(win_totals), color='green', linestyle='--', label=f'Median: {np.median(win_totals):.1f}')

# Add hypothetical o/u line
over_under_line = 45.5
plt.axvline(over_under_line, color='orange', linestyle='-', linewidth=2, label=f'O/U Line: {over_under_line}')

plt.xlabel('Final Win Total')
plt.ylabel('Frequency')
plt.title(f'{team} Win Total Distribution ({len(win_totals):,} Simulations)')
plt.legend()
plt.tight_layout()
plt.show()

# Calculate over probability
over_prob = sum(1 for w in win_totals if w > over_under_line) / len(win_totals)
print(f"\nP(Over {over_under_line}): {over_prob:.1%}")
print(f"P(Under {over_under_line}): {1-over_prob:.1%}")

## 7. Model Confidence Analysis

How confident is the model? Analyze variance across simulations.

In [ ]:
# High confidence = Low variance in outcomes
# Low confidence = High variance

print(f"Win Total Statistics for {team}:")
print(f"  Mean: {np.mean(win_totals):.2f}")
print(f"  Std Dev: {np.std(win_totals):.2f}")
print(f"  Min: {np.min(win_totals)}")
print(f"  Max: {np.max(win_totals)}")
print(f"  95% CI: [{np.percentile(win_totals, 2.5):.1f}, {np.percentile(win_totals, 97.5):.1f}]")

# Interpretation
if np.std(win_totals) < 3:
    print("\n✓ High confidence: Model has strong opinion on team strength")
elif np.std(win_totals) > 5:
    print("\n⚠️ Low confidence: High uncertainty in remaining games")
else:
    print("\n~ Moderate confidence: Typical uncertainty")

## 8. Save Results for Tracking

Save simulation results and bet recommendations for later analysis.

In [ ]:
# Save results
results_dir = Path('../data/simulation_results')
results_dir.mkdir(exist_ok=True)

timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

# Save playoff probabilities
playoff_df.to_csv(results_dir / f'playoff_probs_{timestamp}.csv', index=False)

# Save bet opportunities
if bet_opportunities:
    bet_df = pd.DataFrame(bet_opportunities)
    bet_df.to_csv(results_dir / f'bet_opportunities_{timestamp}.csv', index=False)

print(f"Results saved to {results_dir}/")

## Next Steps

1. **Backtesting**: Use historical data to validate model predictions vs. actual outcomes
2. **Live Tracking**: Run simulations daily and track odds movement
3. **Bet Placement**: When edge > 5% and Kelly > 2%, consider placing bet
4. **Model Improvement**: If perplexity is high, retrain with more data or tune architecture

## Key Insights

**Why This Approach Works:**
- Traditional models give point estimates ("Lakers: 48 wins")
- Our model generates full distributions ("40-50 wins, peaked at 47")
- Books price futures based on public perception + simple models
- Our simulation captures:
  - Schedule strength (who they play)
  - Correlations (if Team A wins, affects Team B's playoff odds)
  - Injuries and form (encoded in context)
  - Momentum effects (learned by model)

**Where to Find Edge:**
- Early-season futures (more uncertainty)
- Mid-tier teams (books focus on contenders)
- Correlated bets (model handles dependencies naturally)
- News-driven odds movement (model incorporates context)

## Performance Metrics to Track

- **ROI**: Return on investment over time
- **Hit Rate**: % of bets that win
- **Calibration**: Do 60% predictions win 60% of the time?
- **Kelly Accuracy**: Are bet sizes optimal?
- **Perplexity**: Lower = better model = better predictions